# Proposed validation — review before it runs

**What this measures:** The base branch cannot import `AbsTopKTrainingSAE`, so both SAEs are trained and scored side by side inside one run (capability, `baseline: none`): the target `explained_variance_gap` (ev_topk − ev_abstopk from the repo's own `run_evals` on held-out tiny-stories-1M batches, identical k/width/seed/tokens) answers "without hurting reconstruction", while `abstopk_negative_share` plus a k-pinned true-nonzero count answer "recovers negative signals". All of it fits CPU and the 30-minute budget at d_model=64, d_sae=1024, k=32, ~2M tokens per arm (SAE step ≈ 2·4096·64·1024·3 ≈ 1.6 GFLOP → well under a second on CPU).

**Target metric:** `explained_variance_gap`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

### Weights & Biases

This validation logs training runs to **Weights & Biases** so the dashboards the repository asks for exist. It needs your W&B API key (W&B → Settings → API keys). In Colab, store it as a secret named `WANDB_API_KEY`; elsewhere set the environment variable. The run goes to entity `smellslikeml` (your W&B username or a team you belong to) and project `remyx-validate-saelens` (the folder the runs are grouped under, created if it does not exist) — the same place the Remyx run logs to, so your run appears beside it. Change the two values below to log somewhere else.

In [ ]:
import os
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
os.environ.setdefault("WANDB_ENTITY", "smellslikeml")
os.environ.setdefault("WANDB_PROJECT", "remyx-validate-saelens")
os.environ.setdefault("WANDB_BASE_URL", "https://api.wandb.ai")
os.environ.setdefault("WANDB_TAGS", "remyx-validate,hand-run")
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set — W&B logging disabled for this run")
else:
    print(f"W&B: {os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_topk_vs_abstopk_tinystories.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_topk_vs_abstopk_tinystories.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
# eval/eval_topk_vs_abstopk_tinystories.py
"""Paired CPU benchmark: TopKTrainingSAE (control) vs AbsTopKTrainingSAE (PR head).

Both architectures are trained on real tiny-stories-1M activations
(blocks.0.hook_mlp_out) through the repo's own LanguageModelSAETrainingRunner with
identical k / d_sae / seed / total tokens (repo W&B logging left ON, online, both arms in one
W&B group per the PR template's "control and test group"), then both are scored through
the repo's run_evals (n_eval_reconstruction_batches=10, n_eval_sparsity_variance_batches=1).

Target: explained_variance_gap = ev_topk - ev_abstopk (user bar: <= 0.02).
Guardrail: topk_negative_share == 0 (TopK ReLUs before top-k, on baseline and head
alike). Diagnostics include the true nonzero (!= 0) selected count per token and the
L0 the repo's evals.py reports. The user's "under 30 minutes" budget is honored by
workload sizing (~2M tokens per arm on CPU), not by a wall-clock metric (tier: cpu).
Prints one JSON line as the LAST line of stdout.
"""
import argparse
import json
import os
import random
import sys
from pathlib import Path

In [ ]:
import numpy as np
import torch
import wandb
from transformer_lens import HookedTransformer

# wandb stays on, online: control + test land in one W&B group; the runner's
# wandb.finish() runs before the final JSON line.
os.environ.setdefault("WANDB_RUN_GROUP", "topk-vs-abstopk-tinystories")

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from sae_lens import (
    ActivationsStore,
    LanguageModelSAERunnerConfig,
    LanguageModelSAETrainingRunner,
    LoggingConfig,
    TopKTrainingSAEConfig,
)
from sae_lens.evals import EvalConfig, run_evals
from sae_lens.training.activation_scaler import ActivationScaler

In [ ]:
# The ONLY defensive import: symbols this diff adds. On the baseline (main) they
# are absent, so we fall back to a TopK-only run and emit zeroed/degraded AbsTopK
# metrics (question.kind = capability; baseline: none).
try:
    from sae_lens import AbsTopKTrainingSAEConfig

    HAS_ABSTOPK = True
except ImportError:
    AbsTopKTrainingSAEConfig = None
    HAS_ABSTOPK = False

In [ ]:
SMOKE = os.environ.get("REMYX_SMOKE", "0") == "1"
SEED = 42
K = 32
MODEL_NAME = "tiny-stories-1M"
DATASET = "apollo-research/roneneldan-tinystories"
HOOK = "blocks.0.hook_mlp_out"
CTX = 64
STORE_BS = 8 if SMOKE else 32  # prompts per store batch -> 512/2048 tokens
TOK = 1024 if SMOKE else 2_000_000  # tokens per arm; sized to fit the 30-min budget
EVAL_RECON_BATCHES = 1 if SMOKE else 10  # evals.py defaults
EVAL_VAR_BATCHES = 1
PROBE_BATCHES = 1 if SMOKE else 8

In [ ]:
model = None
D_IN = D_SAE = None
EVAL_STORE_CFG = None

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

In [ ]:
def make_runner_cfg(sae_cfg):
    return LanguageModelSAERunnerConfig(
        model_name=MODEL_NAME,
        dataset_path=DATASET,
        hook_name=HOOK,
        training_tokens=TOK,
        context_size=CTX,
        train_batch_size_tokens=CTX * STORE_BS,
        store_batch_size_prompts=STORE_BS,
        device="cpu",
        seed=SEED,
        logger=LoggingConfig(  # repo's own logging stays on (online; control + test group)
            log_to_wandb=os.environ.get("WANDB_MODE") != "disabled",
            wandb_entity=os.environ.get("WANDB_ENTITY"),
            wandb_project=os.environ.get("WANDB_PROJECT", "remyx-validate-saelens"),
        ),
        sae=sae_cfg,
    )

In [ ]:
def train_arm(sae_cfg_cls):
    seed_all()  # identical init + token stream for both arms (paired design)
    cfg = make_runner_cfg(sae_cfg_cls(k=K, d_in=D_IN, d_sae=D_SAE))
    return LanguageModelSAETrainingRunner(cfg).run()

@torch.no_grad()
def probe_selected_latents(sae, store, n_batches):
    """Sign/sparsity stats of the latents the SAE actually SELECTS (!= 0, not > 0)."""
    neg = nz = 0
    per_token = []
    alive = None
    for _ in range(n_batches):
        acts = sae.encode(store.next_batch()).reshape(-1, D_SAE)
        nonzero = acts != 0
        neg += int((acts < 0).sum())
        nz += int(nonzero.sum())
        per_token.append(nonzero.sum(-1).float().mean().item())
        alive = nonzero.any(dim=0) if alive is None else alive | nonzero.any(dim=0)
    return {
        "neg_share": neg / max(nz, 1),
        "nonzero_per_token": sum(per_token) / len(per_token),
        "dead": int((~alive).sum()),
    }

In [ ]:
def eval_arm(sae):
    # Fresh, identically-seeded store: both arms are scored on the SAME batches.
    seed_all()
    store = ActivationsStore.from_config(model, EVAL_STORE_CFG)
    results, _ = run_evals(
        sae=sae,
        activation_store=store,
        model=model,
        activation_scaler=ActivationScaler(),
        eval_config=EvalConfig(
            n_eval_reconstruction_batches=EVAL_RECON_BATCHES,
            n_eval_sparsity_variance_batches=EVAL_VAR_BATCHES,
            compute_sparsity_metrics=True,
            compute_variance_metrics=True,
        ),
    )
    return {
        "ev": float(results["reconstruction_quality"]["explained_variance"]),
        "l0": float(results["sparsity"]["l0"]),
        "mse": float(results["reconstruction_quality"]["mse"]),
        **probe_selected_latents(sae, store, PROBE_BATCHES),
    }

In [ ]:
def main():
    global model, D_IN, D_SAE, EVAL_STORE_CFG
    model = HookedTransformer.from_pretrained(MODEL_NAME, device="cpu")
    D_IN = int(model.cfg.d_model)  # tiny-stories-1M: 64, read at runtime
    D_SAE = 16 * D_IN  # 16x expansion per the PR docs' example
    # Any sae cfg with the same dims works for the store (it only streams activations).
    EVAL_STORE_CFG = make_runner_cfg(
        TopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE)
    )

    topk = eval_arm(train_arm(TopKTrainingSAEConfig))
    if HAS_ABSTOPK:
        abstopk = eval_arm(train_arm(AbsTopKTrainingSAEConfig))
    else:  # baseline: capability absent -> degraded (zeroed) AbsTopK metrics
        abstopk = {
            "ev": 0.0,
            "l0": 0.0,
            "mse": 0.0,
            "neg_share": 0.0,
            "nonzero_per_token": 0.0,
            "dead": float(D_SAE),
        }

    metrics = {
        "explained_variance_gap": topk["ev"] - abstopk["ev"],
        "abstopk_negative_share": abstopk["neg_share"],
        "topk_negative_share": topk["neg_share"],
        "abstopk_true_nonzero_per_token": abstopk["nonzero_per_token"],
        "ev_topk": topk["ev"],
        "ev_abstopk": abstopk["ev"],
        "l0_topk": topk["l0"],
        "l0_abstopk": abstopk["l0"],
        "true_nonzero_topk": topk["nonzero_per_token"],
        "mse_topk": topk["mse"],
        "mse_abstopk": abstopk["mse"],
        "dead_latents_topk": topk["dead"],
        "dead_latents_abstopk": abstopk["dead"],
    }
    # Close any open wandb run so its shutdown output precedes the final JSON line.
    wandb.finish()
    print(json.dumps(metrics))

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.parse_args()  # accepted and ignored
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "Does AbsTopK recover negative signals without hurting reconstruction on tiny-stories-1M activations (CPU, same k/width/seed/tokens, under 30 minutes)? Pass if AbsTopK's held-out explained_variance is not worse than TopK's by more than 0.02 and a nonzero share of its selected latents are negative."
loop:
  max_iterations: 8
  fix_code: true
requires: [wandb]
compute:
  tier: cpu
benchmarks:
  - name: topk-vs-abstopk-tinystories
    suite: "eval/eval_topk_vs_abstopk_tinystories.py"
    baseline: none
    packages: [wandb]
    scorer: explained_variance_gap
    policy:
      guardrail_veto: true
    metrics:
      # Every metric below carries an explicit numeric threshold (contract requirement).
      # Diagnostic thresholds are loose structural/derived bounds that a correctly running
      # arm satisfies by construction and only a broken measurement violates; they are
      # reporting bounds, not gating bars (role: diagnostic), and each number's derivation
      # is written next to it.
      #
      # TARGET 1 (user bar): ev_topk - ev_abstopk on the same run_evals held-out batches; pass iff <= 0.02.
      - name: explained_variance_gap
        direction: min
        threshold: 0.02
        role: target
        bar: goal
        reads_as: "how much lower AbsTopK's held-out explained variance is than TopK's; <= 0.02 means not meaningfully worse"
      # TARGET 2 (user bar): at init pre-acts sit near-symmetric about b_enc so ~half of top-|k| picks are negative, and
      # AbsTopK has no non-negativity pressure, so a working sign path leaves well above 1% negative; 0.0 would catch nothing.
      - name: abstopk_negative_share
        direction: max
        threshold: 0.01
        role: target
        bar: goal
        reads_as: "fraction of AbsTopK's selected (nonzero) latent activations that are negative on held-out tokens"
      # GUARDRAIL (the only one, and baseline-achievable): TopK's encode ends in ReLU on both baseline and PR head,
      # so its negative share is exactly 0 on both; > 0 means the selection/probe measurement itself is broken.
      - name: topk_negative_share
        direction: min
        threshold: 0.0
        role: guardrail
        bar: sanity
        reads_as: "negative share among TopK's selected activations; must be exactly 0 because TopK ReLUs before top-k"
      # DIAGNOSTIC, structural bound: top-|k| keeps exactly k=32 entries per token and a selected pre-activation
      # equals 0.0 with probability ~0 in fp32, so the true-nonzero count is exactly 32 -> pass iff >= 32; anything
      # else (0 on a capability-absent run, or a garbled selection) fails. It cannot be a guardrail because the
      # capability-absent fallback zeroes it — a bar the baseline cannot clear would veto every baseline run.
      - name: abstopk_true_nonzero_per_token
        direction: max
        threshold: 32
        role: diagnostic
        reads_as: "mean number of nonzero latents per token for AbsTopK; pinned at exactly k=32 by construction on the PR head (zero on baseline)"
      # DIAGNOSTIC floor: a correctly trained 16x-expansion TopK SAE on 64-dim tiny-stories mlp_out at ~2M tokens
      # reaches held-out EV ~0.9+ in practice; 0.5 is a conservative floor that untrained/degenerate arms (EV ~ 0,
      # e.g. a dead decoder) fall below, so pass iff >= 0.5.
      - name: ev_topk
        direction: max
        threshold: 0.5
        role: diagnostic
        reads_as: "TopK control's held-out explained_variance from the repo's run_evals (10 reconstruction batches)"
      # DIAGNOSTIC floor, same derivation as ev_topk (the 0.02 gap target itself implies ev_abstopk >= ev_topk - 0.02
      # whenever the target passes); pass iff >= 0.5. Zero on a capability-absent run, which is expected to fail here
      # — that is why this is a diagnostic, not a guardrail.
      - name: ev_abstopk
        direction: max
        threshold: 0.5
        role: diagnostic
        reads_as: "AbsTopK's held-out explained_variance from the repo's run_evals (10 reconstruction batches)"
      # DIAGNOSTIC structural bound: encode zeroes all but the k=32 selected entries per token, so however evals.py
      # counts signs the reported L0 cannot exceed 32; > 32 would mean activations leaked past the top-k mask.
      # Pass iff <= 32.
      - name: l0_topk
        direction: min
        threshold: 32
        role: diagnostic
        reads_as: "L0 per token that the repo's evals.py reports for TopK"
      # DIAGNOSTIC structural bound, same as l0_topk: only k=32 of 1024 entries survive top-|k|, so L0 <= 32 whether
      # evals.py counts > 0 (~16 for AbsTopK, whose negative picks are excluded) or != 0 (~32). Pass iff <= 32.
      - name: l0_abstopk
        direction: min
        threshold: 32
        role: diagnostic
        reads_as: "L0 per token that the repo's evals.py reports for AbsTopK; compare against abstopk_true_nonzero_per_token"
      # DIAGNOSTIC structural bound: ReLU-then-top-k keeps at most k=32 nonzero entries per token, so the true
      # nonzero count is <= 32 by construction (it converges up toward 32 as training separates b_enc).
      # Pass iff <= 32; > 32 would mean values leaked past the selection.
      - name: true_nonzero_topk
        direction: min
        threshold: 32
        role: diagnostic
        reads_as: "TopK's true (!= 0) selected count per token, <= k=32, converging to k as training separates b_enc"
      # DIAGNOSTIC ceiling: tiny-stories mlp_out per-feature variance is O(1), and at the EV >= 0.5 floors above a
      # healthy arm sits at MSE ~ 0.05-0.2, so 1.0 is a loose ceiling only a degenerate reconstruction
      # (EV ~ 0, MSE ~ full activation variance) exceeds. Pass iff <= 1.0.
      - name: mse_topk
        direction: min
        threshold: 1.0
        role: diagnostic
        reads_as: "TopK held-out MSE from run_evals"
      # DIAGNOSTIC ceiling, same derivation as mse_topk (per-feature variance O(1) on this 64-dim hook, healthy arms
      # at MSE ~ 0.05-0.2). Pass iff <= 1.0.
      - name: mse_abstopk
        direction: min
        threshold: 1.0
        role: diagnostic
        reads_as: "AbsTopK held-out MSE from run_evals"
      # DIAGNOSTIC ceiling: the probe makes 8 batches x 2048 tokens x k=32 ~= 524k selections across 1024 latents
      # (~512 expected per latent under uniform spread), and the dead-latent aux loss actively drives dead counts
      # toward 0, so a functioning arm stays far below half the dictionary dead; <= 512 (half of d_sae) is a loose
      # ceiling. A collapsed encoder that reuses a few latents leaves ~992 dead and fails. Pass iff <= 512.
      - name: dead_latents_topk
        direction: min
        threshold: 512
        role: diagnostic
        reads_as: "of 1024 latents, those never active across held-out probe batches for TopK (aux selection differs per arm)"
      # DIAGNOSTIC ceiling, same derivation as dead_latents_topk (~524k selections over 1024 latents, aux
      # resurrection, so well under half dead in a functioning arm). Reads 1024 on a capability-absent run, which
      # fails as expected — again why this is a diagnostic, not a guardrail. Pass iff <= 512.
      - name: dead_latents_abstopk
        direction: min
        threshold: 512
        role: diagnostic
        reads_as: "of 1024 latents, those never active across held-out probe batches for AbsTopK (aux selection differs per arm)"
held_constant:
  - "same k=32, d_sae=1024 (16x of d_model=64 read from model.cfg at runtime), seeds reset before each arm, same total_training_tokens and train_batch_size_tokens"
  - "same trainer path (SAETrainer inside LanguageModelSAETrainingRunner, repo logging left on) and same evals.py defaults: n_eval_reconstruction_batches=10, n_eval_sparsity_variance_batches=1"
  - "wandb logging left on for both arms, online, in one WANDB_RUN_GROUP: identical logging on both arms"
  - "same hook point blocks.0.hook_mlp_out on tiny-stories-1M and same dataset through the repo's ActivationsStore for both arms"
avoid:
  - "no synthetic stand-in activations; both arms must train on real tiny-stories-1M activations from the repo's own store"
  - "quality metrics are deterministic quantities (EV, L0, nonzero counts), never wall-clock; the user's 30-minute budget is honored by workload sizing (~2M tokens/arm), not measured, because tier: cpu makes shared timing noise"
  - "do not enable CE/KL evals or move to GPU; the ask is CPU reconstruction parity, not downstream-loss or device claims"
  - "no inline reimplementation of the activation or aux loss; measure only through the repo's TopK/AbsTopK classes and run_evals"
report:
  headline: "On tiny-stories-1M (CPU, matched k/seed/tokens), AbsTopK trains with {abstopk_negative_share:%} of selected latents negative while holding explained variance within {explained_variance_gap} of TopK"
  findings:
    - "selected-latent negative share: AbsTopK {abstopk_negative_share:%} vs TopK {topk_negative_share:%} (0 by construction)"
    - "held-out explained_variance from evals.py: TopK {ev_topk} vs AbsTopK {ev_abstopk}, gap {explained_variance_gap} against the 0.02 bar"
    - "evals.py L0: TopK {l0_topk} vs AbsTopK {l0_abstopk}; true nonzero per token: TopK {true_nonzero_topk} vs AbsTopK {abstopk_true_nonzero_per_token} (k=32)"
    - "held-out MSE: TopK {mse_topk} vs AbsTopK {mse_abstopk}; dead latents of 1024: TopK {dead_latents_topk} vs AbsTopK {dead_latents_abstopk}"
  establishes:
    - "signs survive magnitude-based top-k selection in a really trained SAE, via {abstopk_negative_share} above 0 with the structural count {abstopk_true_nonzero_per_token} pinned at k"
    - "reconstruction parity within the user's 0.02 band on held-out evals.py batches, via {explained_variance_gap}"
  does_not_establish:
    - "the paper's bidirectional concept encoding (probing/steering) or its claim that AbsTopK improves EV; only parity at toy scale is tested"
    - "clean attribution of any EV gap to sign-keeping alone; the dead-neuron aux selection changes jointly (see {dead_latents_abstopk} vs {dead_latents_topk})"
  not_measured:
    - "CE-loss / KL downstream effect (the PR template's CE box), probing or steering accuracy, feature dashboard interpretability"
    - "wall-clock runtime: the user's 30-minute budget is enforced by workload sizing (~2M tokens/arm, ~1000 steps), not measured, since tier: cpu makes wall time noise"
  caveat: "toy-scale proxy: 64-dim tiny-stories-1M, k=32, ~2M tokens per arm on CPU, sized to fit the user's 30-minute budget for both arms; both arms may sit in a high-EV regime where the 0.02 band is easy to meet"
  next: "train longer or at paper scale with compute_ce_loss=True and concept probing to test the bidirectional-feature claim itself"
provenance:
  explained_variance_gap: "user_guidance"
  abstopk_negative_share: "user_guidance"
  topk_negative_share: "inferred"
  abstopk_true_nonzero_per_token: "inferred"
  l0_topk: "user_guidance"
  l0_abstopk: "user_guidance"
  true_nonzero_topk: "user_guidance"
  mse_topk: "pr_template:performance check"
  mse_abstopk: "pr_template:performance check"
  dead_latents_topk: "claim_analysis"
  dead_latents_abstopk: "claim_analysis"
  suite: "synthesized"
  held_constant: "user_guidance"
  requires: "pr_template:wandb dashboards with a control and test group"
  compute: "user_guidance"
logging:
  wandb:
    entity: smellslikeml    # from your W&B connection
    project: remyx-validate-saelens    # default
```